In [1]:
# 1. Import packages
import geopandas as gpd
import pandas as pd
import numpy as np
import folium
import requests
import json
import os
from dotenv import load_dotenv
from shapely.geometry import Point

# 2. Load .env
load_dotenv()

# 3. Load or create shelter data
# 如果你有之前的資料可以改成 read_file()
# gdf_shelters = gpd.read_file("your_file.geojson")

data = [
    ["HL-S001", "花蓮市避難收容點1", "花蓮市", "medium", "low", 18.5, 4.2, Point(305200, 2668200)],
    ["HL-S002", "花蓮市避難收容點2", "花蓮市", "medium", "low", 22.1, 5.7, Point(304600, 2667600)],
    ["HL-S003", "吉安鄉避難收容點1", "吉安鄉", "medium", "medium", 35.4, 8.9, Point(303800, 2662800)],
    ["HL-S004", "吉安鄉避難收容點2", "吉安鄉", "medium", "medium", 41.7, 10.5, Point(302900, 2661800)],
    ["HL-S005", "壽豐鄉避難收容點1", "壽豐鄉", "high", "medium", 92.6, 14.8, Point(299800, 2648600)],
    ["HL-S006", "壽豐鄉避難收容點2", "壽豐鄉", "high", "high", 135.2, 21.3, Point(298900, 2647200)],
    ["HL-S007", "鳳林鎮避難收容點1", "鳳林鎮", "medium", "medium", 76.8, 12.6, Point(292700, 2630400)],
    ["HL-S008", "光復鄉避難收容點1", "光復鄉", "medium", "medium", 88.1, 11.9, Point(287300, 2612600)],
    ["HL-S009", "瑞穗鄉避難收容點1", "瑞穗鄉", "high", "medium", 109.4, 16.7, Point(282700, 2594300)],
    ["HL-S010", "玉里鎮避難收容點1", "玉里鎮", "medium", "medium", 72.3, 9.8, Point(277900, 2571800)],
    ["HL-S011", "新城鄉避難收容點1", "新城鄉", "medium", "low", 28.9, 6.1, Point(304200, 2683400)],
    ["HL-S012", "秀林鄉避難收容點1", "秀林鄉", "high", "high", 245.6, 31.4, Point(296300, 2675200)],
    ["HL-S013", "富里鄉避難收容點1", "富里鄉", "medium", "medium", 64.5, 7.6, Point(274800, 2552800)],
]

columns = [
    "shelter_id", "name", "TOWNNAME",
    "risk_level", "terrain_risk",
    "mean_elevation", "max_slope",
    "geometry"
]

gdf_shelters = gpd.GeoDataFrame(data, columns=columns, crs="EPSG:3826")

# 4. Print info
print("Shelter count:", len(gdf_shelters))
print("CRS:", gdf_shelters.crs)
print("Columns:", gdf_shelters.columns)

Shelter count: 13
CRS: EPSG:3826
Columns: Index(['shelter_id', 'name', 'TOWNNAME', 'risk_level', 'terrain_risk',
       'mean_elevation', 'max_slope', 'geometry'],
      dtype='str')


In [2]:
def fetch_cwa_api(api_key):
    url = "https://opendata.cwa.gov.tw/api/v1/rest/datastore/O-A0002-001"
    params = {
        "Authorization": api_key,
        "format": "JSON"
    }

    response = requests.get(url, params=params)

    if response.status_code == 200:
        return response.json()
    else:
        print("API request failed:", response.status_code)
        return None

In [3]:
def parse_rainfall_json(data):
    stations = []

    try:
        station_data = data['records']['Station']
    except:
        print("Invalid JSON structure")
        return gpd.GeoDataFrame()

    for s in station_data:
        try:
            name = s['StationName']
            lat = float(s['GeoInfo']['Coordinates'][1]['StationLatitude'])
            lon = float(s['GeoInfo']['Coordinates'][1]['StationLongitude'])

            rainfall = None
            for r in s['RainfallElement']:
                if r['ElementName'] == 'RAIN':
                    rainfall = float(r['ElementValue'])
            
            if rainfall == -998:
                continue

            stations.append({
                "name": name,
                "rainfall": rainfall,
                "geometry": Point(lon, lat)
            })
        except:
            continue

    gdf = gpd.GeoDataFrame(stations, crs="EPSG:4326")
    return gdf

In [4]:
MODE = "SIMULATION"  # 或 "LIVE"

if MODE == "LIVE":
    api_key = os.getenv("CWA_API_KEY")
    data = fetch_cwa_api(api_key)
else:
    # 模擬資料
    data = {
        "records": {
            "Station": [
                {
                    "StationName": "Test1",
                    "GeoInfo": {
                        "Coordinates": [
                            {},
                            {"StationLatitude": "23.99", "StationLongitude": "121.60"}
                        ]
                    },
                    "RainfallElement": [
                        {"ElementName": "RAIN", "ElementValue": "120"}
                    ]
                }
            ]
        }
    }

gdf_rain = parse_rainfall_json(data)
print("Rain stations:", len(gdf_rain))

Rain stations: 1


In [5]:
# 投影轉換
gdf_shelters_wgs84 = gdf_shelters.to_crs(epsg=4326)

m = folium.Map(location=[23.8, 121.5], zoom_start=8)

In [6]:
for _, row in gdf_rain.iterrows():
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=5,
        popup=f"{row['name']}: {row['rainfall']} mm",
        color="blue"
    ).add_to(m)

In [7]:
def color_by_risk(risk):
    if risk == "high":
        return "red"
    elif risk == "medium":
        return "orange"
    else:
        return "green"

for _, row in gdf_shelters_wgs84.iterrows():
    folium.Marker(
        location=[row.geometry.y, row.geometry.x],
        popup=row['name'],
        icon=folium.Icon(color=color_by_risk(row['risk_level']))
    ).add_to(m)

In [8]:
m

m.save("map.html")